# Explicação do motor autodiff

**As derivadas parciais que fazem uma rede neural aprender.**

Este caderno parte da derivada de Cálculo I, passa pelas derivadas parciais de Cálculo II e chega ao mecanismo que treina redes neurais: a **diferenciação automática** (em inglês, *autodiff*). O motor usado aqui foi escrito do zero, em Python puro, no arquivo `rede_neural/value.py`.

O caminho:

1. Revisão de Cálculo I: a derivada
2. O problema: um neurônio e o seu erro
3. Derivadas parciais
4. Descida do gradiente
5. Três jeitos de calcular derivadas no computador
6. A regra da cadeia com várias variáveis
7. O motor: o grafo computacional
8. A volta: retropropagação
9. Laboratório
10. Do motor à rede neural

## Glossário

Os termos aparecem na ordem do caderno. No começo, cada ideia é apresentada por um apelido. Mais adiante, ela ganha o nome que tem em Cálculo.

| nome | apelido no começo | em uma frase |
|---|---|---|
| razão incremental | quanto a saída andou ÷ quanto a entrada andou | $\dfrac{f(x+h) - f(x)}{h}$ |
| derivada | | o limite da razão incremental quando $h \to 0$: a inclinação da reta tangente |
| regra da cadeia | as taxas se multiplicam | a derivada de uma composição é o produto das derivadas de cada etapa |
| neurônio | | palpite $= w \cdot x + b$ |
| função de várias variáveis | o erro, que depende dos botões | $E(w, b)$: um número que depende de mais de uma entrada |
| parâmetros | os botões | $w$ e $b$, as variáveis que o treino ajusta |
| derivada parcial | mexer só num botão | $\dfrac{\partial E}{\partial w}$: a derivada em relação a $w$, com $b$ constante |
| traço | o corte da superfície | a curva obtida cortando o gráfico com um plano, como $b = b_0$ |
| vetor gradiente | sentir com os pés para onde o chão desce | $\nabla E = \left(\dfrac{\partial E}{\partial w}, \dfrac{\partial E}{\partial b}\right)$: aponta para onde $E$ cresce mais rápido |
| curva de nível | o mapa de relevo | os pontos $(w, b)$ em que $E$ tem o mesmo valor |
| descida do gradiente | andar contra a derivada | $(w, b) \leftarrow (w, b) - \eta\, \nabla E$ |
| taxa de aprendizado | o tamanho do passo | o $\eta$ da descida do gradiente |
| diferenciação numérica, simbólica e automática | os três jeitos de derivar | derivar pela definição, pela fórmula ou pelo grafo das contas |
| regra da cadeia para várias variáveis | os caminhos se somam | com vários caminhos até uma variável, as contribuições de cada caminho se somam |
| grafo computacional | as caixinhas e setas | o diagrama de árvore das contas: cada nó é uma conta |
| derivada parcial local | a derivada da própria conta | a derivada de um nó em relação às suas entradas |
| retropropagação | a volta | aplicar a regra da cadeia do fim do grafo para o começo |
| aproximação linear | a previsão | $\Delta E \approx \dfrac{\partial E}{\partial w}\Delta w + \dfrac{\partial E}{\partial b}\Delta b$ |

In [34]:
import sys
sys.path.insert(0, "../rede_neural")    # a pasta onde mora o motor

from value import Value                 # o motor autodiff
from grafo_visual import (desenha, derivada_interativa, superficie_interativa,
                          descida_no_plano, passo_a_passo, laboratorio, ajuste_interativo)

## Aonde vamos chegar

O computador vai receber quatro exemplos de uma regra escondida:

| entrada | $-1$ | $0$ | $1$ | $2$ |
|---|:-:|:-:|:-:|:-:|
| resposta certa | $0$ | $2$ | $4$ | $6$ |

Ninguém conta a regra para ele. A reta no desenho é o palpite do computador: no começo ela está deitada e erra tudo.

Aperte o botão ▶︎ e veja a reta se ajustar aos pontos sozinha. Até o fim do caderno, cada pedaço do que acontece aqui vai ter explicação.

In [35]:
exemplos = [(-1, 0), (0, 2), (1, 4), (2, 6)]    # (entrada, resposta certa)

ajuste_interativo(Value, exemplos)

## 1. Revisão de Cálculo I: a derivada

Uma derivada responde a uma pergunta só:

> **Se eu mexer um pouquinho na entrada, quanto a saída muda?**

Pegue $f(x) = x^2$ no ponto $x = 3$, onde $f(3) = 9$. Agora dê um empurrão na entrada, para $3{,}1$. A saída vai para $f(3{,}1) = 9{,}61$. A entrada andou $0{,}1$ e a saída andou $0{,}61$: a saída andou **6,1 vezes** o que a entrada andou.

Com empurrões cada vez menores, essa razão vai chegando perto de um número:

| empurrão $h$ | $f(3+h)$ | a saída andou | saída ÷ entrada |
|:-:|:-:|:-:|:-:|
| $1$ | $16$ | $7$ | $7$ |
| $0{,}1$ | $9{,}61$ | $0{,}61$ | $6{,}1$ |
| $0{,}01$ | $9{,}0601$ | $0{,}0601$ | $6{,}01$ |
| $0{,}001$ | $9{,}006001$ | $0{,}006001$ | $6{,}001$ |

Esse número, $6$, é a **derivada** de $f$ no ponto $x = 3$:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

Em palavras: *quanto a saída andou, dividido por quanto a entrada andou, quando o empurrão fica tão pequeno quanto se queira.*

> **Em Cálculo I:** o empurrão $h$ é o **acréscimo**, a razão $\dfrac{f(x+h) - f(x)}{h}$ é a **razão incremental**, e a reta que passa pelos dois pontos é a **reta secante**. Quando $h \to 0$, a secante vira a **reta tangente**, e a derivada é o **coeficiente angular** dela.

### Da secante à tangente

Diminua o $h$ no slider: o segundo ponto escorrega até o primeiro e a reta secante (azul) vai encostando na reta tangente (vermelha, tracejada).

In [36]:
derivada_interativa(Value, lambda x: x**2, x0=3)

### O sinal da derivada diz o lado

- **Derivada positiva:** aumentar a entrada faz a saída **subir**.
- **Derivada negativa:** aumentar a entrada faz a saída **descer**.
- **Derivada zero:** naquele ponto a curva está plana, como no fundo de um vale ou no topo de um morro.

O tamanho diz o quanto. Derivada $6$ quer dizer que, perto dali, a saída anda cerca de 6 vezes o que a entrada andou.

> **Guarde esta ideia: é o sinal da derivada que vai dizer para a rede neural para que lado girar cada botão.**

### As regras que vamos usar

Ninguém calcula o limite toda vez. A definição vira regras prontas, e o motor inteiro só precisa destas:

| função | derivada | exemplo |
|---|---|---|
| uma constante $c$ | $0$ | $(5)' = 0$ |
| $c \cdot x$ | $c$ | $(3x)' = 3$ |
| $x^n$ | $n \cdot x^{n-1}$ (regra da potência, a do tombo) | $(x^2)' = 2x$ |
| $u + v$ | $u' + v'$ | $(x^2 + 3x)' = 2x + 3$ |
| $u - v$ | $u' - v'$ | $(x^2 - 3x)' = 2x - 3$ |

### Funções dentro de funções: as taxas se multiplicam

Um carro anda 2 vezes mais rápido que uma bicicleta. A bicicleta anda 3 vezes mais rápido que uma pessoa a pé. Então o carro anda $2 \times 3 = 6$ vezes mais rápido que a pessoa.

Funções dentro de funções funcionam do mesmo jeito. Pegue $f(x) = (2x - 6)^2$ e chame o que está dentro do parêntese de $u = 2x - 6$. O caminho de $x$ até $f$ tem duas etapas:

- quanto $u$ muda quando $x$ muda: $\dfrac{du}{dx} = 2$
- quanto $f$ muda quando $u$ muda: $\dfrac{df}{du} = 2u$

A taxa do caminho inteiro é o produto das etapas:

$$\frac{df}{dx} = \frac{df}{du} \cdot \frac{du}{dx} = 2u \cdot 2$$

Em $x = 0$, temos $u = -6$, então $f'(0) = (-12) \cdot 2 = -24$.

> **Em Cálculo I:** isso é a **regra da cadeia**, $(f \circ g)'(x) = f'(g(x)) \cdot g'(x)$. Cada etapa só precisa saber a própria taxa, e a taxa do caminho inteiro é o produto das etapas.

Essa frase vai virar o motor autodiff inteiro.

## 2. O problema: um neurônio e o seu erro

### Aprender é ajustar botões

Começando pelo caso mais simples, com um único exemplo de treino: quando a entrada é $x = 2$, a resposta certa é $y = 6$.

O "cérebro" do computador é uma fórmula com dois botões ajustáveis, o **peso** $w$ e o **viés** $b$:

$$\text{palpite} = w \cdot x + b$$

Isso é um **neurônio**: ele multiplica a entrada por um peso e soma um viés. Uma rede neural é um monte de neurônios ligados entre si, cada um com os seus botões. Modelos como o ChatGPT têm bilhões de botões.

Os botões começam zerados. Com $w = 0$ e $b = 0$, o palpite para $x = 2$ é $0$. A resposta certa era $6$.

### Medindo o erro

Para melhorar, primeiro é preciso medir o quanto errou:

$$E = (\text{palpite} - \text{resposta certa})^2 = (w x + b - y)^2$$

O quadrado faz três coisas: o erro fica sempre positivo (errar para cima ou para baixo conta igual), erros grandes pesam muito mais que erros pequenos, e a curva fica lisa, sem bicos, o que permite derivar.

Com os botões zerados: $E = (0 - 6)^2 = 36$.

### Aprender é deixar o erro o menor possível

Com $x = 2$ e $y = 6$ fixos, o erro só depende dos dois botões:

$$E(w, b) = (2w + b - 6)^2$$

Agora "aprender" tem um significado preciso: **encontrar os valores de $w$ e $b$ que deixam $E$ o menor possível.**

> **Em Cálculo II:** $E(w, b)$ é uma **função de duas variáveis**, e os botões $w$ e $b$ são as suas **variáveis independentes**. Em redes neurais, eles se chamam **parâmetros**. O gráfico de $E$ deixa de ser uma curva e vira uma **superfície** no espaço, e treinar a rede é procurar o ponto mais baixo dessa superfície.

Daqui em diante, os botões são parâmetros.

## 3. Derivadas parciais

Com dois parâmetros, a pergunta da seção 1 se divide em duas:

- quanto o erro muda se eu mexer **só** em $w$?
- quanto o erro muda se eu mexer **só** em $b$?

Cada resposta é uma **derivada parcial**. A definição é a mesma de Cálculo I. O empurrão vai em uma variável, e a outra fica parada:

$$\frac{\partial E}{\partial w} = \lim_{h \to 0} \frac{E(w + h,\, b) - E(w,\, b)}{h} \qquad\qquad \frac{\partial E}{\partial b} = \lim_{h \to 0} \frac{E(w,\, b + h) - E(w,\, b)}{h}$$

Na prática, a conta é a de Cálculo I com uma regra a mais:

> **A variável que não está sendo mexida é tratada como constante.**

O símbolo $\partial$ é um "d" arredondado que avisa que a derivada é parcial. $\dfrac{\partial E}{\partial w}$ se lê *derivada parcial de $E$ em relação a $w$*, e também se escreve $E_w$.

Exemplo: $f(w, b) = 3w + b^2$.

- Mexendo só em $w$, o $b^2$ é constante, e constante tem derivada zero: $f_w = 3$.
- Mexendo só em $b$, o $3w$ é constante: $f_b = 2b$.

A regra resolve a multiplicação sem esforço. Em $f = a \cdot b$, mexendo só em $a$, o $b$ é constante, como o $3$ em $3a$. Então $f_a = b$ e, do mesmo jeito, $f_b = a$. **Cada fator recebe o valor do outro.**

### As derivadas parciais do erro

Com a regra da cadeia da seção 1:

$$\frac{\partial E}{\partial w} = 2(2w + b - 6) \cdot 2 \qquad\qquad \frac{\partial E}{\partial b} = 2(2w + b - 6) \cdot 1$$

Com os parâmetros zerados, $2w + b - 6 = -6$. Então $\dfrac{\partial E}{\partial w} = -24$ e $\dfrac{\partial E}{\partial b} = -12$. As duas são negativas: aumentar $w$ ou $b$ diminui o erro. Faz sentido, porque o palpite ($0$) ficou abaixo da resposta ($6$).

### No gráfico: cortes da superfície

Fixar $b = b_0$ é cortar a superfície com o plano $b = b_0$. O que sobra é uma curva comum, de uma variável só, e $\partial E/\partial w$ é a inclinação da reta tangente a essa curva. Fixar $w = w_0$ corta na outra direção, e $\partial E/\partial b$ é a inclinação desse outro corte.

> **Em Cálculo II:** cada corte é um **traço** da superfície. A derivada parcial $\partial E/\partial w$ é o coeficiente angular da reta tangente ao traço no plano $b = b_0$, e $\partial E/\partial b$ é o da reta tangente ao traço no plano $w = w_0$.

Gire a superfície e mova o ponto $P$. Repare no fundo: não é um ponto, é uma calha inteira. Todos os $(w, b)$ com $2w + b = 6$ acertam o único exemplo.

In [37]:
superficie_interativa(Value, [(2, 6)])

### Para que lado andar?

O treino pode mexer em $w$ e $b$ ao mesmo tempo, em qualquer direção do plano. Imagine alguém no alto de um morro com neblina, sem enxergar o vale. Dá para sentir com os pés para que lado o chão desce. As derivadas parciais são esse "sentir com os pés", uma direção de cada vez:

- $\partial E/\partial w = -24$: andando no sentido em que $w$ cresce, o erro desce. Então $w$ deve **aumentar**.
- $\partial E/\partial b = -12$: o mesmo vale para $b$, com metade da força.

Andar **contra** as derivadas leva morro abaixo. Juntando as duas num vetor, $\nabla E = \left(\dfrac{\partial E}{\partial w},\ \dfrac{\partial E}{\partial b}\right) = (-24,\ -12)$, e a direção que desce mais rápido é $-\nabla E = (24,\ 12)$.

> **Em Cálculo II:** $\nabla E$ é o **vetor gradiente** de $E$. Ele aponta para onde $E$ cresce mais rápido, e $-\nabla E$ aponta para onde $E$ desce mais rápido.

## 4. Descida do gradiente

A seção 3 deu a resposta: para o erro cair o mais rápido possível, o passo vai na direção $-\nabla E$.

$$(w, b)_{\text{novo}} = (w, b) - \eta\, \nabla E \qquad\text{ou seja}\qquad w \leftarrow w - \eta\,\frac{\partial E}{\partial w}, \qquad b \leftarrow b - \eta\,\frac{\partial E}{\partial b}$$

O $\eta$ (letra grega eta) controla o tamanho do passo.

> **Em Cálculo II (e em otimização):** esse é o **método da descida do gradiente**, e $\eta$ é a **taxa de aprendizado**. O "andar contra a derivada" de antes é andar na direção $-\nabla E$, a de maior decrescimento.

### No mapa de curvas de nível

O desenho mostra a superfície vista de cima. Cada linha é uma **curva de nível**: os pontos $(w, b)$ com o mesmo erro, como num mapa de relevo.

Aperte ▶︎ e acompanhe o caminho, que sai de $(0, 0)$:

- Com **1 exemplo**, o caminho vai reto, perpendicular às curvas de nível, e para em $(2{,}4;\ 1{,}2)$, em cima da reta de mínimos. Com taxa $0{,}1$, ele chega lá num passo só.
- Troque para **4 exemplos** (os do começo). As curvas de nível viram elipses e o fundo vira um ponto só, $(2, 2)$. Com taxa $0{,}13$, o caminho faz zigue-zague: cada passo é perpendicular à curva de nível, e não aponta direto para o fundo.
- Com taxas grandes demais, o passo pula o fundo e o erro explode: com 4 exemplos, a partir de $0{,}2$. Com 1 exemplo, a taxa $0{,}2$ fica pulando de um lado para o outro para sempre.

In [38]:
quatro_exemplos = [(-1, 0), (0, 2), (1, 4), (2, 6)]

descida_no_plano(Value, {"1 exemplo": [(2, 6)], "4 exemplos": quatro_exemplos})

## 5. Três jeitos de calcular derivadas no computador

Numa rede de verdade, $\nabla E$ tem milhões ou bilhões de componentes, uma para cada parâmetro, e a descida do gradiente precisa dele **a cada passo** do treino. Há três jeitos de pôr o computador para calcular derivadas:

| jeito | nome | como funciona | problema |
|---|---|---|---|
| pela definição | **diferenciação numérica** | empurra cada parâmetro um pouquinho e mede quanto $E$ mudou, como na tabela da seção 1 | uma conta inteira de $E$ **para cada parâmetro**: com um bilhão deles, um bilhão de contas por passo. E como $h$ nunca é zero, o resultado é aproximado |
| como no caderno | **diferenciação simbólica** | aplica as regras de derivação na fórmula inteira de $E$ | numa rede grande, as fórmulas das derivadas ficam gigantescas e repetem os mesmos pedaços milhares de vezes |
| pelo grafo das contas | **diferenciação automática** | anota cada conta pequena enquanto calcula $E$ e depois aplica a regra da cadeia nessas anotações | nenhum: todas as derivadas parciais, exatas, pelo custo de mais ou menos uma conta extra de $E$ |

A diferenciação automática tem dois modos. O **modo direto** carrega as derivadas junto com as contas, da entrada para a saída, e precisa de uma passada para cada parâmetro. O **modo reverso** volta da saída para as entradas e entrega o gradiente inteiro numa passada só. Com um único erro $E$ e bilhões de parâmetros, o modo reverso ganha, e é ele que este motor implementa.

## 6. A regra da cadeia com várias variáveis

Na seção 1, cada etapa dependia de uma só etapa anterior. Em contas de verdade, uma variável pode influenciar o resultado **por mais de um caminho**.

O caso clássico: $z = f(x, y)$, com $x = x(t)$ e $y = y(t)$. Mexer em $t$ muda $z$ por dois caminhos, passando por $x$ e passando por $y$:

$$\frac{dz}{dt} = \frac{\partial z}{\partial x}\,\frac{dx}{dt} + \frac{\partial z}{\partial y}\,\frac{dy}{dt}$$

> **Em Cálculo II:** essa é a **regra da cadeia para funções de várias variáveis**. Ao longo de cada caminho as derivadas se multiplicam, e as contribuições dos caminhos se somam. O desenho de quem depende de quem se chama **diagrama de árvore**, e cada variável é um **nó** dele.

### Um exemplo pequeno

Seja $f = a \cdot b + a$, com $a = 3$ e $b = 2$. O $a$ chega até $f$ por dois caminhos:

- passando pela multiplicação $u = a \cdot b$: $\dfrac{\partial f}{\partial u} \cdot \dfrac{\partial u}{\partial a} = 1 \cdot b = 2$;
- direto, pela soma: $1$.

Somando: $\dfrac{\partial f}{\partial a} = 2 + 1 = 3$.

O motor desenha o diagrama de árvore sozinho, deitado, da esquerda para a direita. Os números $\times k$ nas setas são as derivadas parciais de cada etapa. Seguindo de $f$ até $a$: pelo caminho direto, $\times 1$; pelo caminho da multiplicação, $\times 1$ e depois $\times 2$. Somando, $1 + 1 \cdot 2 = 3$.

In [39]:
a, b = Value(3.0), Value(2.0)
f = a*b + a
f.backward()

print("∂f/∂a =", a.grad)
print("∂f/∂b =", b.grad)
desenha(f, f=f, a=a, b=b)

∂f/∂a = 3.0
∂f/∂b = 3.0


## 7. O motor: o grafo computacional

### Quebrando a conta em pedaços

O computador não enxerga $(wx + b - y)^2$ de uma vez. Ele faz uma conta pequena por vez, cada uma com no máximo dois números:

| ordem | conta | resultado, com $x = 2$, $y = 6$, $w = 0$, $b = 0$ |
|:-:|---|:-:|
| 1 | $p = w \cdot x$ | $0$ |
| 2 | $q = p + b$ | $0$ |
| 3 | $u = q - y$ | $-6$ |
| 4 | $E = u^2$ | $36$ |

Desenhando cada conta como uma caixinha, ligada por setas às caixinhas que ela usou, aparece o diagrama de árvore da seção 6.

> **Em computação:** esse diagrama se chama **grafo computacional**. Cada caixinha é um **nó**, e cada seta, uma **aresta**.

### O nó no código: a classe `Value`

Cada nó é um objeto da classe `Value`. Ele guarda:

- `data`: o número calculado (em azul no desenho);
- `grad`: a derivada parcial do erro final em relação a este nó, $\partial E / \partial(\text{nó})$. Começa em $0$ e é preenchida depois (em vermelho);
- `_prev`: de quais nós ele veio (as arestas);
- `_op`: qual conta o criou (o símbolo no círculo);
- `_backward`: uma pequena função que sabe a derivada daquela conta. É ela que trabalha na volta.

O truque: quando o Python encontra `w * x` e os dois são `Value`, ele chama um método especial da classe, o `__mul__`. Foi nele que o motor ensinou ao Python o que fazer: calcular o produto, criar um nó novo e anotar de onde ele veio. O mesmo vale para `+` (`__add__`), `-` (`__sub__`) e `**` (`__pow__`). Por isso o código abaixo parece uma conta comum, mas por baixo ele está montando o grafo.

In [40]:
x = Value(2.0)    # entrada
y = Value(6.0)    # resposta certa
w = Value(0.0)    # peso: parâmetro
b = Value(0.0)    # viés: parâmetro

E = (w*x + b - y) ** 2

desenha(E, parametros=("w", "b"), E=E, x=x, y=y, w=w, b=b)

Os números azuis foram calculados da esquerda para a direita. Essa é a **ida** (em inglês, *forward pass*). Todo `grad` ainda está em $0$, porque a volta não começou.

## 8. A volta: retropropagação

### Cada nó só conhece a derivada da própria conta

Cada tipo de nó sabe a derivada da sua conta em relação a cada entrada. São as regras da seção 1, com a regra das derivadas parciais da seção 3:

| conta | em relação a $a$ | em relação a $b$ | onde está no código |
|---|:-:|:-:|---|
| $a + b$ | $1$ | $1$ | `__add__` |
| $a - b$ | $1$ | $-1$ | `__sub__` |
| $a \cdot b$ | $b$ | $a$ | `__mul__` |
| $a^n$ | $n \cdot a^{n-1}$ | | `__pow__` |

> **Em Cálculo II:** a "derivada da própria conta" é uma **derivada parcial local**: a derivada de uma única função intermediária em relação às suas entradas, sem olhar o resto do grafo.

### Juntando com a regra da cadeia

Para descobrir o gradiente de um nó, multiplica-se o gradiente que chegou do nó seguinte pela derivada parcial local. É a regra da cadeia:

$$\underbrace{\frac{\partial E}{\partial a}}_{\texttt{a.grad}} \;=\; \underbrace{\frac{\partial E}{\partial\,\text{saída}}}_{\texttt{out.grad}} \times \text{derivada parcial local}$$

No código, toda operação tem essa mesma linha, só trocando a derivada local. Na multiplicação, por exemplo:

```python
def _backward():
    self.grad  += out.grad * other.data    # a derivada de a·b em relação a a é b
    other.grad += out.grad * self.data     # e em relação a b é a
```

### Por onde começa: pelo fim

A derivada de $E$ em relação a ele mesmo é $1$: se $E$ sobe um pouquinho, $E$ sobe exatamente esse pouquinho. Então a volta começa com `E.grad = 1` e, a partir daí, as derivadas vão sendo passadas para trás, nó por nó.

> **Em redes neurais:** essa volta se chama **retropropagação** (em inglês, *backpropagation*).

Clique em **Próximo passo** e acompanhe. O círculo destacado é a conta que está rodando, e o amarelo marca o que acabou de mudar.

In [41]:
passo_a_passo(Value, lambda x, y, w, b: (w*x + b - y) ** 2,
              parametros=("w", "b"), x=2.0, y=6.0, w=0.0, b=0.0)

### Tudo de uma vez: `backward()`

O método `backward()` faz exatamente esses passos, sozinho, numa linha só:

In [42]:
E.backward()

print("∂E/∂w =", w.grad)
print("∂E/∂b =", b.grad)
desenha(E, parametros=("w", "b"), E=E, x=x, y=y, w=w, b=b)

∂E/∂w = -24.0
∂E/∂b = -12.0


São os mesmos $-24$ e $-12$ da seção 3, calculados à mão.

E o `x.grad = 0`? Não é erro. Com $w = 0$, o produto $w \cdot x$ dá zero para qualquer $x$, então mexer em $x$ não muda nada no erro.

Um detalhe: rodar a célula de cima duas vezes dobra os gradientes, porque eles se somam (a segunda regra logo abaixo explica). Para repetir, rode antes a célula da seção 7, que monta o grafo de novo.

### Duas regras do jogo

**A ordem importa.** Um nó só pode passar a derivada para trás depois de receber a parte de **todos** os nós que o usam. É como uma corrente de recados: ninguém repassa antes de ouvir todo mundo que fala com ele. Por isso o `backward()` primeiro organiza os nós numa fila em que cada um aparece depois de tudo de que depende (é o trabalho da função `constroi`) e depois percorre essa fila de trás para frente. Essa fila se chama **ordem topológica**.

**As derivadas se somam.** É a regra da cadeia para várias variáveis da seção 6: quando um nó é usado em dois lugares, a derivada chega por dois caminhos e as contribuições se somam. Por isso o código usa `+=` em vez de `=`, e por isso o `grad` começa em $0$. No passo a passo de $f = a \cdot b + a$, dá para ver o $a$ recebendo uma parte de cada vez.

In [43]:
passo_a_passo(Value, lambda a, b: a*b + a, nome_saida="f", a=3.0, b=2.0)

## 9. Laboratório

Aqui dá para mexer em tudo. Cada slider muda um valor, e o motor refaz a ida e a volta inteiras na hora. O amarelo marca o que mudou.

- **Dar um passo** aplica uma vez a descida do gradiente nos parâmetros $w$ e $b$.
- Na tabela das derivadas parciais, cada linha usa a derivada para **prever** quanto o erro muda se aquele valor subir $0{,}1$, e compara com o que acontece **de verdade**.

> **Em Cálculo II:** essa previsão é a **aproximação linear**, ou diferencial: $\Delta E \approx \dfrac{\partial E}{\partial w}\,\Delta w$ quando só $w$ muda, e $\Delta E \approx \dfrac{\partial E}{\partial w}\,\Delta w + \dfrac{\partial E}{\partial b}\,\Delta b$ quando os dois mudam. Ela é boa para passos pequenos, e é por isso que a descida do gradiente usa passos pequenos.

Para testar:

- Deixe $w = 0$ e arraste o $x$. A derivada parcial em relação a $x$ não sai do zero. Por quê? (A resposta está na seção 8.)
- Ajuste $w$ e $b$ até o palpite $w \cdot x + b$ ficar igual a $y$. O erro vai a zero e todas as derivadas parciais zeram: é o fundo do vale.
- Perto do fundo do vale, a aproximação linear e o valor de verdade começam a discordar. A derivada só enxerga a inclinação, e perto do fundo a superfície já está entortando.
- Coloque $x = 3$, taxa $0{,}19$ e treine 20 passos: o erro explode. Com $x = 2$ e a mesma taxa, ele converge. O tamanho seguro do passo depende do formato do vale.

In [44]:
laboratorio(Value, lambda x, y, w, b: (w*x + b - y) ** 2,
            x=2.0, y=6.0, w=0.0, b=0.0)

### Sem sliders

O mesmo, escrito direto. Mude os números, rode a célula e confira com as contas à mão.

In [45]:
x = Value(2.0)
y = Value(6.0)
w = Value(0.0)
b = Value(0.0)

E = (w*x + b - y) ** 2
E.backward()

desenha(E, parametros=("w", "b"), E=E, x=x, y=y, w=w, b=b)

## 10. Do motor à rede neural

### O treino em quatro tempos

Lembra da reta do começo? É isto que acontece por trás dela. Cada volta do laço tem os quatro tempos de qualquer treino de rede neural:

1. **Ida:** calcula o palpite e o erro de todos os exemplos. O motor monta o grafo.
2. **Zerar:** apaga as derivadas da rodada anterior. Como elas se somam, sobraria lixo.
3. **Volta:** `backward()` calcula $\nabla E$, a derivada parcial do erro em relação a cada parâmetro.
4. **Ajuste:** um passo de descida do gradiente, $(w, b) \leftarrow (w, b) - \eta\,\nabla E$.

Com vários exemplos, o erro total é a soma dos erros de cada um. Quem faz essa soma é o próprio motor, então cada parâmetro recebe derivada de todos os exemplos de uma vez, pelo mesmo `+=` da seção 8.

In [46]:
exemplos = [(-1, 0), (0, 2), (1, 4), (2, 6)]    # (entrada, resposta certa)

w, b = Value(0.0), Value(0.0)
taxa = 0.05

for passo in range(41):
    E = sum((w*x + b - y) ** 2 for x, y in exemplos)    # 1. ida
    w.grad, b.grad = 0.0, 0.0                            # 2. zerar
    E.backward()                                         # 3. volta
    if passo % 10 == 0:
        print(f"passo {passo:2d}   erro = {E.data:8.4f}   w = {w.data:.3f}   b = {b.data:.3f}")
    w.data -= taxa * w.grad                              # 4. ajuste
    b.data -= taxa * b.grad

print(f"\nRegra encontrada: resposta = {w.data:.2f} · entrada + {b.data:.2f}")
print(f"Palpite para uma entrada nova, 10: {w.data * 10 + b.data:.2f}")

passo  0   erro =  56.0000   w = 0.000   b = 0.000
passo 10   erro =   0.0018   w = 2.013   b = 1.978
passo 20   erro =   0.0000   w = 2.001   b = 1.999
passo 30   erro =   0.0000   w = 2.000   b = 2.000
passo 40   erro =   0.0000   w = 2.000   b = 2.000

Regra encontrada: resposta = 2.00 · entrada + 2.00
Palpite para uma entrada nova, 10: 22.00


O treino chegou em $w \approx 2$ e $b \approx 2$, o fundo do vale com os 4 exemplos (o mesmo ponto da seção 4). A regra escondida era resposta $= 2 \cdot$ entrada $+ 2$, e ninguém contou isso para o computador. Tudo saiu de derivadas parciais.

### O que muda numa rede de verdade

Nada no mecanismo. Só o tamanho do grafo.

- **Mais entradas por neurônio.** Um neurônio com várias entradas calcula $w_1 x_1 + w_2 x_2 + \dots + b$: as mesmas somas e multiplicações, em maior número, e $E$ vira uma função de muitas variáveis. O grafo abaixo é um neurônio com duas entradas, montado com o mesmo motor.
- **Uma função de ativação.** Sem ela, uma rede inteira, por maior que seja, só consegue desenhar retas. A ativação "dobra" a reta e deixa a rede aprender curvas. Para o motor, é só mais um tipo de nó com a sua derivada local.
- **Camadas.** As saídas de um grupo de neurônios viram as entradas do grupo seguinte. O grafo fica maior, mas a volta é a mesma.

Bibliotecas como **PyTorch** e **TensorFlow**, usadas para treinar as redes neurais de verdade, fazem exatamente isso por baixo. A diferença é que, em cada nó, em vez de um número há uma tabela inteira de números.

In [47]:
x1, x2 = Value(1.5), Value(-0.5)     # duas entradas
w1, w2 = Value(0.8), Value(-1.2)     # um peso para cada entrada
b, y = Value(0.3), Value(1.0)        # viés e resposta certa

E = (w1*x1 + w2*x2 + b - y) ** 2
E.backward()

desenha(E, parametros=("w1", "w2", "b"), E=E, x1=x1, x2=x2, w1=w1, w2=w2, b=b, y=y)

---

**Em resumo:** cada conta anota de onde veio, cada nó conhece só a sua derivada parcial local, e uma passada de volta junta tudo pela regra da cadeia para várias variáveis. Treinar uma rede neural é repetir isso milhares de vezes, num grafo muito maior.

### O Cálculo II que apareceu

- **Funções de várias variáveis:** o erro depende de todos os parâmetros ao mesmo tempo.
- **Derivadas parciais:** quanto o erro muda por parâmetro.
- **Traços da superfície:** a inclinação de cada corte é uma derivada parcial.
- **Vetor gradiente:** por que andar na direção $-\nabla E$ faz o erro cair.
- **Curvas de nível:** o mapa do erro visto de cima.
- **Regra da cadeia para várias variáveis:** o motor autodiff inteiro.
- **Aproximação linear:** por que passos pequenos funcionam.